## XGBoost

In [ ]:
# Python ≥3.9 is required
import sys
assert sys.version_info >= (3, 9)

# Scikit-Learn ≥1.0 is required
import sklearn
assert sklearn.__version__ >= "1.0"

# Common imports
import numpy as np
import pandas as pd
import os

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Precision options
np.set_printoptions(precision=2)
pd.options.display.float_format = '{:.3f}'.format

# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")
# ignore convergence warnings for the purpose of this class
warnings.filterwarnings(action="ignore", category=sklearn.exceptions.ConvergenceWarning)

In [ ]:
from google.colab import files
import os

In [ ]:
os.makedirs("data", exist_ok=True)

In [ ]:
uploaded = files.upload()

Saving train_bin_nhanes.csv to train_bin_nhanes.csv


In [ ]:
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")

In [ ]:
train_import = pd.read_csv('data/train_bin_nhanes.csv')

In [ ]:
uploaded = files.upload()

Saving test_bin_nhanes.csv to test_bin_nhanes.csv


In [ ]:
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")

In [ ]:
test_import = pd.read_csv('data/test_bin_nhanes.csv')

In [ ]:
columns_to_remove = [
    "LBXGLU",
    "URDACT",
    "RIDAGEYR",
    "BMXBMI",
    "LBDHDD",
    "INDFMPIR",
    "LBXIN",
    "SLD012",
    "LBXTLG",
    "SLD013"
]

train_import = train_import.drop(columns=columns_to_remove)
test_import = test_import.drop(columns=columns_to_remove)

In [ ]:
X_train = train_import.drop(columns=["target"])
y_train = train_import["target"]

X_test = test_import.drop(columns=["target"])
y_test = test_import["target"]

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(y_train.unique())

(5372, 56)
(5372,)
[0 1]


In [ ]:
# Display scores function
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", f"{scores.mean():.2f}")
    print("Standard deviation:", f"{scores.std():.2f}")

In [ ]:
#reduced to 5 to reduce time
from sklearn.model_selection import cross_validate, KFold
n_splits = 5
k_fold = KFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

xgb_extreme = XGBClassifier(
    n_estimators=100,
    max_depth=None,
    learning_rate=0.6,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=77,
    n_jobs=-1
)

xgb_extreme.fit(X_train, y_train)

# Training performance
y_train_pred_xgb_extreme = xgb_extreme.predict(X_train)
y_train_probs_xgb_extreme = xgb_extreme.predict_proba(X_train)[:, 1]

train_auc_xgb_extreme = roc_auc_score(
    y_train,
    y_train_probs_xgb_extreme
)

# Hold-out performance
y_test_pred_xgb_extreme = xgb_extreme.predict(X_test)
y_test_probs_xgb_extreme = xgb_extreme.predict_proba(X_test)[:, 1]

test_auc_xgb_extreme = roc_auc_score(
    y_test,
    y_test_probs_xgb_extreme
)

print("Extreme Overfitting")
print("Training ROC-AUC:", train_auc_xgb_extreme)
print("Hold-out ROC-AUC:", test_auc_xgb_extreme)


Extreme Overfitting
Training ROC-AUC: 0.9999287964246407
Hold-out ROC-AUC: 0.7690899823927348


In [ ]:
from sklearn.utils import resample
import numpy as np

def bootstrap_optimism_auc(model, X_train, y_train, B=50):

    apparent_scores = []
    test_scores = []
    optimisms = []

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    for i in range(B):

        # Bootstrap sample
        X_boot, y_boot = resample(
            X_train,
            y_train,
            replace=True
        )

        # Fit model on bootstrap sample
        model.fit(X_boot, y_boot)

        # Bootstrap performance
        boot_probs = model.predict_proba(X_boot)[:, 1]

        # Performance on original training data
        full_probs = model.predict_proba(X_train)[:, 1]

        boot_auc = roc_auc_score(
            y_boot,
            boot_probs
        )

        test_auc = roc_auc_score(
            y_train,
            full_probs
        )

        optimism = boot_auc - test_auc

        apparent_scores.append(boot_auc)
        test_scores.append(test_auc)
        optimisms.append(optimism)

    avg_optimism = np.mean(optimisms)

    # Final model fitted to full training data
    model.fit(X_train, y_train)

    apparent_final = roc_auc_score(
        y_train,
        model.predict_proba(X_train)[:, 1]
    )

    corrected = apparent_final - avg_optimism

    return {
        "apparent": apparent_final,
        "optimism": avg_optimism,
        "corrected": corrected
    }

In [ ]:
xgb_extreme_bootstrap = bootstrap_optimism_auc(
    xgb_extreme,
    X_train,
    y_train,
    B=50
)

print("Extreme Overfitting - Bootstrap")
print("Apparent AUC:", xgb_extreme_bootstrap["apparent"])
print("Optimism:", xgb_extreme_bootstrap["optimism"])
print("Corrected AUC:", xgb_extreme_bootstrap["corrected"])


Extreme Overfitting - Bootstrap
Apparent AUC: 0.9999287964246407
Optimism: 0.08318281327593802
Corrected AUC: 0.9167459831487027


In [ ]:
xgb_moderate = XGBClassifier(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.01,
    subsample=0.1,
    colsample_bytree=0.3,
    min_child_weight=5,
    reg_lambda=100,
    reg_alpha=5,
    gamma=5,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=77,
    n_jobs=-1
)

xgb_moderate.fit(X_train, y_train)

# Training performance
y_train_pred_xgb_moderate = xgb_moderate.predict(X_train)
y_train_probs_xgb_moderate = xgb_moderate.predict_proba(X_train)[:, 1]

train_auc_xgb_moderate = roc_auc_score(
    y_train,
    y_train_probs_xgb_moderate
)

# Hold-out performance
y_test_pred_xgb_moderate = xgb_moderate.predict(X_test)
y_test_probs_xgb_moderate = xgb_moderate.predict_proba(X_test)[:, 1]

test_auc_xgb_moderate = roc_auc_score(
    y_test,
    y_test_probs_xgb_moderate
)

print("No Overfitting")
print("Training ROC-AUC:", train_auc_xgb_moderate)
print("Hold-out ROC-AUC:", test_auc_xgb_moderate)


Moderate Overfitting
Training ROC-AUC: 0.7140516822296479
Hold-out ROC-AUC: 0.722187471040682


In [ ]:
xgb_moderate_bootstrap = bootstrap_optimism_auc(
    xgb_moderate,
    X_train,
    y_train,
    B=50
)

print("No Overfitting - Bootstrap")
print("Apparent AUC:", xgb_moderate_bootstrap["apparent"])
print("Optimism:", xgb_moderate_bootstrap["optimism"])
print("Corrected AUC:", xgb_moderate_bootstrap["corrected"])


Moderate Overfitting - Bootstrap
Apparent AUC: 0.7140516822296479
Optimism: 0.0032726178203207777
Corrected AUC: 0.7107790644093271
